<a href="https://colab.research.google.com/github/magooly/dogecoin/blob/1.4.1/Copy_of_MACD_Tradingbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Be careful with an execution of this script. The MACD alone is ->probably<-
#not a good indicator to trade alone.
# Please read the disclaimer below the video!
#Video instructions: https://youtu.be/lNvJXKXUQ_U

In [ ]:
from binance.client import Client
import pandas as pd
import ta

In [ ]:
#%run ./Binance_Keys.ipynb
client = Client(api_key,api_secret)

In [ ]:
from time import sleep

In [ ]:
from binance.exceptions import BinanceAPIException

In [ ]:
def getminutedata(symbol):
    try:
        df = pd.DataFrame(client.get_historical_klines(symbol, '1m', '40m UTC'))
    except BinanceAPIException as e:
        print(e)
        sleep(60)
        df = pd.DataFrame(client.get_historical_klines(symbol, '1m', '40m UTC'))
    df = df.iloc[:,:6]
    df.columns = ['Time','Open','High','Low','Close','Volume']
    df = df.set_index('Time')
    df.index = pd.to_datetime(df.index, unit='ms')
    df = df.astype(float)
    return df

In [ ]:
def tradingstrat(symbol, qty, open_position = False):
    while True:
        df = getminutedata(symbol)
        if not open_position:
            if ta.trend.macd_diff(df.Close).iloc[-1] > 0 \
            and ta.trend.macd_diff(df.Close).iloc[-2] < 0:
                order = client.create_order(symbol=symbol,
                                           side='BUY',
                                           type='MARKET', quantity=qty)
                print(order)
                open_position = True
                buyprice = float(order['fills'][0]['price'])
                break
    if open_position:
        while True:
            df = getminutedata(symbol)
            if ta.trend.macd_diff(df.Close).iloc[-1] < 0 \
            and ta.trend.macd_diff(df.Close).iloc[-2] > 0:
                order = client.create_order(symbol=symbol,
                                           side='SELL',
                                           type='MARKET', quantity=qty)
                print(order)
                sellprice = float(order['fills'][0]['price'])
                print(f'profit = {(sellprice - buyprice)/buyprice}')
                open_position = False
                break

In [ ]:
while True:
    tradingstrat('ETHUSDT', qty=0.1)

BinanceAPIException: APIError(code=-2010): Account has insufficient balance for requested action.